# Dynamic Fintech Workloads — Dependencies, Conditions and Iteration

**Objective:** Extend our ingestion workflow with dependent joins, conditional duplicate handling, and state-based credit application reports.

## Why?

**For our project:** Combine staged customers, credit applications, and installments, then create reusable processing tasks and reports.

**For Data Engineering:** Control execution order, pass runtime task values, branch based on data quality, and reuse one parameterized notebook for multiple states.

#### 01 — Verify Existing Staging Tables

Read the catalog and schema parameters, then inspect the staging table schemas and record counts to confirm that the inputs are ready for the dependent processing tasks.

In [ ]:
# Define defaults for manual notebook execution.
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "fintech_jobs")

# Read the current parameter values.
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
namespace = f"{catalog}.{schema}"

# Inspect the existing staging tables.
table_names = [
    "customers_staging",
    "applications_staging",
    "installments_staging",
]

for table_name in table_names:
    full_table_name = f"{namespace}.{table_name}"
    df = spark.table(full_table_name)

    print(f"{full_table_name}: {df.count()} records")
    df.printSchema()

workspace.fintech_jobs.customers_staging: 5 records
root
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- state: string (nullable = true)
 |-- registration_date: date (nullable = true)

workspace.fintech_jobs.applications_staging: 6 records
root
 |-- application_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- application_date: date (nullable = true)
 |-- requested_amount: decimal(12,2) (nullable = true)
 |-- status: string (nullable = true)

workspace.fintech_jobs.installments_staging: 6 records
root
 |-- installment_id: string (nullable = true)
 |-- application_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- due_date: string (nullable = true)
 |-- amount_due: string (nullable = true)
 |-- payment_status: string (nullable = true)



#### 02 — Configure the Dynamic Fintech Job

Configure `fintech_credit_dynamic_job` to orchestrate ingestion, joins, conditional duplicate handling, and state-based reporting.

Use task dependencies to wait for required inputs, an If/else condition to control duplicate removal, and a For each task to generate application reports for each state.

#### 03 — Configure Conditional Duplicate Handling

Use the `has_duplicates` value published by `customers_installments_summary` to determine whether duplicate removal is required.

A True result selects duplicate removal. A False result proceeds directly to transformation.

#### 04 — Connect the Conditional Branches

When duplicates are detected, remove them before transformation.
Otherwise, proceed directly to transformation.

The shared transformation task uses `None failed` to continue
after either branch, provided no dependency has failed.

#### 05 — Configure State Report Iteration

Use a For each task to generate application reports for PE, SP,
RJ, MG, and BA with the same notebook.

Pass each state to the nested notebook through its `state`
parameter. Set concurrency to 1 to process states sequentially.

#### 06 — Execute the Dynamic Fintech Job

Run the complete workflow with `simulate_duplicate=false`.

Verify that the condition selects the False branch, transformation
completes without duplicate removal, and all five state iterations
succeed.

For a controlled True-branch test, override `simulate_duplicate`
to `true` for a single Job run. Expect 7 joined rows, 6 distinct
rows, and one duplicate removed before transformation.
Keep the saved Job default as `false`.

The controlled True-branch test passed in run `<RECORDED_ID_06>`.
Duplicate detection identified 7 rows with 6 distinct records.
Duplicate removal removed exactly one row, transformation produced
6 curated records, and all five state iterations succeeded.